# Task 01: Data Cleaning and Preprocessing Pipeline

**Internship Project:** Data Science Internship  
**Project:** Comprehensive Data Cleaning and Preprocessing in Python  
**Author:** Data Science Intern  
**Tools & Libraries:** Python, Pandas, NumPy, Jupyter Notebook  

---

### Table of Contents
1. [A. Import Required Libraries](#A.-Import-Required-Libraries)
2. [B. Load the Raw CSV Dataset](#B.-Load-the-Raw-CSV-Dataset)
3. [C. Display First 5 and Last 5 Rows](#C.-Display-First-5-and-Last-5-Rows)
4. [D. Check Dataset Shape](#D.-Check-Dataset-Shape)
5. [E. Check Column Names](#E.-Check-Column-Names)
6. [F. Check Data Types Using df.info()](#F.-Check-Data-Types-Using-df.info())
7. [G. Generate Basic Statistical Information Using df.describe()](#G.-Generate-Basic-Statistical-Information-Using-df.describe())
8. [H. Identify Missing Values Using df.isnull().sum()](#H.-Identify-Missing-Values-Using-df.isnull().sum())
9. [I. Handle Missing Values Using Appropriate Techniques](#I.-Handle-Missing-Values-Using-Appropriate-Techniques)
10. [J. Identify and Remove Duplicate Records](#J.-Identify-and-Remove-Duplicate-Records)
11. [K. Detect and Correct Inconsistent Categorical Values](#K.-Detect-and-Correct-Inconsistent-Categorical-Values)
12. [L. Correct Incorrect Data Types Where Necessary](#L.-Correct-Incorrect-Data-Types-Where-Necessary)
13. [M. Handle Invalid Numerical Values Appropriately](#M.-Handle-Invalid-Numerical-Values-Appropriately)
14. [N. Check the Dataset Again After Cleaning](#N.-Check-the-Dataset-Again-After-Cleaning)
15. [O. Compare Dataset Before and After Cleaning](#O.-Compare-Dataset-Before-and-After-Cleaning)
16. [P. Save the Final Cleaned Dataset](#P.-Save-the-Final-Cleaned-Dataset)
17. [Data Cleaning Summary](#Data-Cleaning-Summary)


---
## A. Import Required Libraries

Before beginning any data science operation, we must import the core libraries:
* **Pandas**: The primary library for data manipulation, analysis, and tabular data structures (DataFrames).
* **NumPy**: Provides support for multidimensional arrays, mathematical functions, and handling `NaN` values.
* **os**: Allows interacting with the operating system to construct reliable relative file paths.


In [1]:
import os
import pandas as pd
import numpy as np

# Adjust display settings for readable DataFrame inspection in Jupyter
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")
print("Core libraries loaded successfully!")


Pandas Version: 3.0.6
NumPy Version:  2.5.3
Core libraries loaded successfully!


---
## B. Load the Raw CSV Dataset

We load the raw employee CSV dataset into a Pandas DataFrame using `pd.read_csv()`.
To ensure this notebook runs seamlessly whether executed from the project root directory or the `notebooks/` directory, we implement dynamic path resolution.

We also keep a pristine copy named `df_raw` so that we can conduct a strict **Before vs. After** comparison at the conclusion of the notebook.


In [2]:
# Dynamically resolve file path to support running from either project root or notebooks/ folder
raw_data_path = '../data/raw_dataset.csv' if os.path.exists('../data/raw_dataset.csv') else 'data/raw_dataset.csv'

# Load the raw dataset
df_raw = pd.read_csv(raw_data_path)

# Create an active working copy for data cleaning
df = df_raw.copy()

print(f"Dataset successfully loaded from: {raw_data_path}")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")


Dataset successfully loaded from: ../data/raw_dataset.csv
Loaded 420 rows and 9 columns.


---
## C. Display First 5 and Last 5 Rows

Examining the head and tail of a dataset provides immediate visual cues regarding:
* Formatting inconsistencies (e.g., extra whitespaces around names, departments, and cities)
* Missing values (`NaN`)
* Formats of dates and monetary fields (e.g., dollar signs and commas in salaries)


In [3]:
print("=== FIRST 5 ROWS (df.head()) ===")
df.head()


=== FIRST 5 ROWS (df.head()) ===


,Employee_ID,Full_Name,Department,Age,Gender,City,Joining_Date,Salary,Experience_Years
0,EMP001,Sneha Sharma,Sales,36.0,Male,Chennai,2016-11-24,"$110,000",11.0
1,EMP002,Rohan Sharma,Marketing,36.0,F,pune,2015.09.07,"$63,000",11.0
2,EMP003,Rekha Reddy,marketing,49.0,Female,MUMBAI,25-04-2017,"$48,000",27.0
3,EMP004,Suresh Mehta,Human Resources,NaN,m,delhi,2022.09.04,"$45,000",34.0
4,EMP005,Rajesh Das,Finance,34.0,male,delhi,2018/05/03,"$145,000",12.0


In [4]:
print("=== LAST 5 ROWS (df.tail()) ===")
df.tail()


=== LAST 5 ROWS (df.tail()) ===


,Employee_ID,Full_Name,Department,Age,Gender,City,Joining_Date,Salary,Experience_Years
415,EMP322,Swati Bose,engineering,52.0,m,Bengaluru,27-06-2015,"$74,000",28.0
416,EMP270,Kunal Patel,Human Resources,42.0,Male,Bangalore,2024/07/12,"$132,000",17.0
417,EMP168,Kavita Chopra,Sales,42.0,Female,Bangalore,2017-06-17,"$75,000",19.0
418,EMP246,Karan Saxena,sales,53.0,NaN,Pune,2020.10.13,"$110,000",31.0
419,EMP230,Naveen Pandey,FINANCE,149.0,Female,Bangalore,2015.05.14,NaN,35.0


---
## D. Check Dataset Shape

The `.shape` attribute returns a tuple representing the dimensionality of the DataFrame: `(rows, columns)`.
This gives us our baseline record count before cleaning.


In [5]:
rows, cols = df.shape
print(f"Number of Rows:    {rows}")
print(f"Number of Columns: {cols}")


Number of Rows:    420
Number of Columns: 9


---
## E. Check Column Names

Reviewing column names ensures that columns are intuitive, correctly spelled, and adhere to consistent conventions (such as snake_case or PascalCase).


In [6]:
print("Dataset Columns:")
for idx, col in enumerate(df.columns, start=1):
    print(f"  {idx}. {col}")


Dataset Columns:
  1. Employee_ID
  2. Full_Name
  3. Department
  4. Age
  5. Gender
  6. City
  7. Joining_Date
  8. Salary
  9. Experience_Years


---
## F. Check Data Types Using df.info()

`df.info()` reveals:
1. The total number of entries and index range.
2. The count of non-null values per column (indicating missing data immediately).
3. The assigned data type (`dtype`) of each column.

**Initial Observations:**
* `Joining_Date` is stored as `object` (string) rather than `datetime64`.
* `Salary` is stored as `object` (string) because of `$` and `,` characters.
* Several columns show fewer non-null entries than the total row count, indicating missing values.


In [7]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Employee_ID       420 non-null    str    
 1   Full_Name         415 non-null    str    
 2   Department        401 non-null    str    
 3   Age               398 non-null    float64
 4   Gender            402 non-null    str    
 5   City              397 non-null    str    
 6   Joining_Date      420 non-null    str    
 7   Salary            393 non-null    str    
 8   Experience_Years  401 non-null    float64
dtypes: float64(2), str(7)
memory usage: 29.7 KB


---
## G. Generate Basic Statistical Information Using df.describe()

`df.describe()` calculates key descriptive statistics:
* **count**: Number of non-null records
* **mean**: Average value
* **std**: Standard deviation (spread of data)
* **min / max**: Minimum and maximum boundaries
* **25%, 50% (median), 75%**: Quartiles

We also use `df.describe(include=['object'])` to inspect categorical columns.

**Key Anomalies Detected:**
* `Age` exhibits a negative minimum (e.g., `-25.0`) and an unrealistic maximum (e.g., `150.0`).
* `Salary` cannot yet be described numerically because it is stored as text.


In [8]:
print("=== NUMERICAL COLUMNS SUMMARY ===")
df.describe()


=== NUMERICAL COLUMNS SUMMARY ===


,Age,Experience_Years
count,398.000000,401.000000
mean,42.216080,17.214464
std,15.280535,11.257393
min,-45.000000,-3.000000
25%,32.000000,6.000000
50%,43.000000,18.000000
75%,51.000000,27.000000
max,149.000000,35.000000


In [9]:
print("=== CATEGORICAL COLUMNS SUMMARY ===")
df.describe(include=['object'])


=== CATEGORICAL COLUMNS SUMMARY ===


C:\Users\Anuj\AppData\Local\Temp\ipykernel_18516\2560209820.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=['object'])


,Employee_ID,Full_Name,Department,Gender,City,Joining_Date,Salary
count,420,415,401,402,397,420,393
unique,400,375,20,11,24,392,115
top,EMP051,Simran Nair,marketing,m,delhi,2023-01-13,"$111,000"
freq,2,3,29,45,32,3,8


---
## H. Identify Missing Values Using df.isnull().sum()

Missing data occurs due to human omission, collection failure, or system errors.
We quantify missing values by calculating:
1. The absolute count of missing values per column.
2. The percentage of missing data relative to the total number of records.


In [10]:
# Compute missing values count and percentage
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': missing_count.values,
    'Missing Percentage (%)': missing_pct.round(2).values
}).sort_values(by='Missing Count', ascending=False).reset_index(drop=True)

print(f"Total Missing Values Across Dataset: {missing_count.sum()}")
missing_summary


Total Missing Values Across Dataset: 133


,Column,Missing Count,Missing Percentage (%)
0,Salary,27,6.43
1,City,23,5.48
2,Age,22,5.24
3,Experience_Years,19,4.52
4,Department,19,4.52
5,Gender,18,4.29
6,Full_Name,5,1.19
7,Employee_ID,0,0.00
8,Joining_Date,0,0.00


---
## I. Handle Missing Values Using Appropriate Techniques

Selecting the right imputation strategy is critical in data science:

### Statistical Rationale for Chosen Techniques:
1. **Full_Name (`object`)**:
   * *Method*: Replace missing values with `'Unknown Employee'`.
   * *Why*: Person names cannot be mathematically computed or averaged. A designated placeholder preserves the record while clearly flagging unrecorded identity.

2. **Department, City, Gender (`categorical`)**:
   * *Method*: Impute using the **Mode** (`df[col].mode()[0]`).
   * *Why*: The mode represents the most frequent category. For categorical variables, mode imputation replaces missing entries with the most statistically probable class without introducing artificial categories.

3. **Age, Experience_Years (`numerical`)**:
   * *Method*: Impute using the **Median** (`df[col].median()`).
   * *Why*: The **mean** is highly vulnerable to extreme outliers (such as negative ages or age 150), which pull the average. The **median** represents the 50th percentile (middle value) and is robust and resistant to outliers.

4. **Salary (`numerical / string`)**:
   * Missing salaries will be filled with the median salary once the column is converted to numeric format in Step L.


In [11]:
# 1. Fill missing Full_Name with placeholder
df['Full_Name'] = df['Full_Name'].fillna("Unknown Employee")
print("[Imputed] 'Full_Name': Replaced missing values with 'Unknown Employee'")

# 2. Impute Categorical Columns using Mode
categorical_cols = ['Department', 'Gender', 'City']
for col in categorical_cols:
    col_mode = df[col].mode()[0]
    df[col] = df[col].fillna(col_mode)
    print(f"[Imputed] '{col}': Replaced missing values with Mode ('{col_mode}')")

# 3. Impute Numerical Columns using Median
numerical_cols = ['Age', 'Experience_Years']
for col in numerical_cols:
    col_median = df[col].median()
    df[col] = df[col].fillna(col_median)
    print(f"[Imputed] '{col}': Replaced missing values with Median ({col_median})")

# Check remaining nulls in handled columns
print("\nRemaining nulls in handled columns:")
print(df[['Full_Name', 'Department', 'Gender', 'City', 'Age', 'Experience_Years']].isnull().sum())


[Imputed] 'Full_Name': Replaced missing values with 'Unknown Employee'
[Imputed] 'Department': Replaced missing values with Mode ('marketing')
[Imputed] 'Gender': Replaced missing values with Mode ('m')
[Imputed] 'City': Replaced missing values with Mode ('delhi')
[Imputed] 'Age': Replaced missing values with Median (43.0)
[Imputed] 'Experience_Years': Replaced missing values with Median (18.0)

Remaining nulls in handled columns:
Full_Name           0
Department          0
Gender              0
City                0
Age                 0
Experience_Years    0
dtype: int64


---
## J. Identify and Remove Duplicate Records

Duplicate rows occur frequently when combining data sources or due to repeated form submissions.
Keeping duplicate records:
* Artificially inflates sample sizes
* Introduces sampling bias into statistics and machine learning models

We identify duplicates using `df.duplicated()`, inspect a sample, and drop them using `df.drop_duplicates()`.


In [12]:
# 1. Check duplicate count
num_duplicates = df.duplicated().sum()
print(f"Number of exact duplicate rows found: {num_duplicates}")

# Display sample duplicate rows
if num_duplicates > 0:
    print("\nSample Duplicate Rows:")
    sample_dupes = df[df.duplicated(keep=False)].sort_values(by='Employee_ID').head(6)
    print(sample_dupes[['Employee_ID', 'Full_Name', 'Department', 'City']])


Number of exact duplicate rows found: 20

Sample Duplicate Rows:
    Employee_ID           Full_Name   Department     City
50       EMP051    Suresh Chopra       marketing     pune
409      EMP051    Suresh Chopra       marketing     pune
69       EMP070     Ananya Kulkarni   Marketing   Chennai
401      EMP070     Ananya Kulkarni   Marketing   Chennai
72       EMP073        Sunita Singh  engineering     pune
414      EMP073        Sunita Singh  engineering     pune


In [13]:
# 2. Remove duplicates and reset index
df = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicate rows remaining after removal: {df.duplicated().sum()}")
print(f"New dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")


Duplicate rows remaining after removal: 0
New dataset shape: 400 rows, 9 columns


---
## K. Detect and Correct Inconsistent Categorical Values

Categorical inconsistency is a major cause of data fragmentation. For instance, `"Delhi"`, `"delhi"`, and `" Delhi "` would be treated by Python as three distinct cities.

### Inconsistencies Targeted:
1. **Leading and Trailing Spaces**: Extra spaces around text entries.
2. **Casing Discrepancies**: Mixed lowercase, uppercase, and title-cased entries.
3. **Aliases and Abbreviations**:
   * City: `"Bangalore"` vs `"Bengaluru"`
   * Department: `"HR"` / `"Hr"` vs `"Human Resources"`
   * Gender: `"M"` / `"m"` vs `"Male"`, `"F"` / `"f"` vs `"Female"`


In [14]:
print("=== BEFORE STANDARDIZATION ===")
print("Unique Departments:", df['Department'].unique().tolist())
print("Unique Cities:     ", df['City'].unique().tolist())
print("Unique Genders:    ", df['Gender'].unique().tolist())


=== BEFORE STANDARDIZATION ===
Unique Departments: ['Sales', ' Marketing ', 'marketing', 'Human Resources', ' Finance ', 'HR', 'Marketing', 'Sales ', 'Finance', 'sales', 'ENGINEERING', ' Engineering ', 'Hr', ' Human Resources ', 'engineering', 'FINANCE', 'Engineering', 'finance', 'hr', ' SALES ']
Unique Cities:      [' Chennai ', 'pune', 'MUMBAI', 'delhi', ' Bengaluru ', 'mumbai', ' Mumbai ', ' Bangalore ', 'bangalore', ' Delhi', 'Mumbai', 'hyderabad', ' Delhi ', ' Hyderabad ', 'chennai', 'Chennai', 'DELHI', 'Bangalore', 'Hyderabad', 'Pune', 'bengaluru', 'Bengaluru', 'Delhi', ' Pune ']
Unique Genders:     [' Male ', 'F', 'Female', 'm', 'male', 'female', ' Female ', 'f', 'M', 'Male', 'Other']


In [15]:
# 1. Strip whitespaces and standardize casing to Title Case across text columns
for col in ['Full_Name', 'Department', 'City', 'Gender']:
    df[col] = df[col].astype(str).str.strip().str.title()

# 2. Standardize Department abbreviations
df['Department'] = df['Department'].replace({
    'Hr': 'Human Resources'
})

# 3. Standardize City aliases
df['City'] = df['City'].replace({
    'Bangalore': 'Bengaluru'
})

# 4. Standardize Gender abbreviations
df['Gender'] = df['Gender'].replace({
    'M': 'Male',
    'F': 'Female'
})

print("=== AFTER STANDARDIZATION ===")
print("Unique Departments:", sorted(df['Department'].unique().tolist()))
print("Unique Cities:     ", sorted(df['City'].unique().tolist()))
print("Unique Genders:    ", sorted(df['Gender'].unique().tolist()))


=== AFTER STANDARDIZATION ===
Unique Departments: ['Engineering', 'Finance', 'Human Resources', 'Marketing', 'Sales']
Unique Cities:      ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Mumbai', 'Pune']
Unique Genders:     ['Female', 'Male', 'Other']


---
## L. Correct Incorrect Data Types Where Necessary

Storing columns in inappropriate data types prevents statistical computation and chronological analysis.

### Corrections Performed:
1. **`Joining_Date`**:
   * *Current Type*: `object` (string with mixed date patterns)
   * *Target Type*: `datetime64[ns]`
   * *Rationale*: Enables date arithmetic, tenure calculation, and chronological filtering.
2. **`Salary`**:
   * *Current Type*: `object` (string formatted with `$` and `,`)
   * *Target Type*: `float64`
   * *Rationale*: Removing non-numeric symbols permits arithmetic operations, aggregation, and statistical modeling. Missing salaries are imputed with the median.
3. **`Age` & `Experience_Years`**:
   * *Target Type*: `int64`
   * *Rationale*: Whole years are discrete quantities.


In [16]:
# 1. Convert Joining_Date to datetime
df['Joining_Date'] = pd.to_datetime(df['Joining_Date'], format='mixed')
print(f"'Joining_Date' converted to: {df['Joining_Date'].dtype}")

# 2. Clean and convert Salary
# Remove currency symbol '$' and commas ','
df['Salary'] = (
    df['Salary']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Impute missing salaries with the median salary of valid positive entries
median_salary = df.loc[df['Salary'] > 0, 'Salary'].median()
df['Salary'] = df['Salary'].fillna(median_salary)
print(f"'Salary' converted to: {df['Salary'].dtype} (Missing values filled with median: ${median_salary:,.2f})")

# 3. Convert Age and Experience to integer
df['Age'] = df['Age'].round().astype(int)
df['Experience_Years'] = df['Experience_Years'].round().astype(int)

print("\nData Types After Type Correction:")
print(df.dtypes)


'Joining_Date' converted to: datetime64[us]
'Salary' converted to: float64 (Missing values filled with median: $94,000.00)

Data Types After Type Correction:
Employee_ID                    str
Full_Name                      str
Department                     str
Age                          int64
Gender                         str
City                           str
Joining_Date        datetime64[us]
Salary                     float64
Experience_Years             int64
dtype: object


---
## M. Handle Invalid Numerical Values Appropriately

Raw data often contains logically or physically impossible records due to data entry errors:
1. **`Age`**: An employee's age cannot be negative (`< 18`) or biologically unrealistic (`> 65`).
   * *Correction*: Detect anomalies and replace them with the median valid age.
2. **`Salary`**: Full-time employees cannot have non-positive salaries (`<= 0`).
   * *Correction*: Detect anomalies and replace them with the median positive salary.
3. **`Experience_Years`**:
   * Experience cannot be negative (`< 0`).
   * Experience cannot exceed `(Age - 18)` (an employee cannot gain professional experience before adulthood).
   * *Correction*: Replace negative experience with 0, and cap excessive experience at `(Age - 18)`.


In [17]:
# 1. Handle Invalid Age values
invalid_age_mask = (df['Age'] < 18) | (df['Age'] > 65)
print(f"Invalid Age records detected (<18 or >65): {invalid_age_mask.sum()}")

if invalid_age_mask.sum() > 0:
    print("Sample invalid age records:")
    print(df.loc[invalid_age_mask, ['Employee_ID', 'Full_Name', 'Age']])
    
    # Calculate median of valid ages
    valid_age_median = int(df.loc[~invalid_age_mask, 'Age'].median())
    df.loc[invalid_age_mask, 'Age'] = valid_age_median
    print(f"Replaced invalid Age records with median: {valid_age_median}")

# 2. Handle Invalid Salary values (<= 0)
invalid_sal_mask = df['Salary'] <= 0
print(f"\nInvalid Salary records detected (<= 0): {invalid_sal_mask.sum()}")

if invalid_sal_mask.sum() > 0:
    print("Sample invalid salary records:")
    print(df.loc[invalid_sal_mask, ['Employee_ID', 'Full_Name', 'Salary']])
    
    valid_sal_median = df.loc[~invalid_sal_mask, 'Salary'].median()
    df.loc[invalid_sal_mask, 'Salary'] = valid_sal_median
    print(f"Replaced non-positive Salary records with median: ${valid_sal_median:,.2f}")

# 3. Handle Invalid Experience values
neg_exp_mask = df['Experience_Years'] < 0
print(f"\nNegative Experience records detected: {neg_exp_mask.sum()}")
if neg_exp_mask.sum() > 0:
    df.loc[neg_exp_mask, 'Experience_Years'] = 0
    print("Replaced negative Experience records with 0")

# Logical consistency check: Experience cannot exceed Age - 18
max_possible_exp = (df['Age'] - 18).clip(lower=0)
exceed_exp_mask = df['Experience_Years'] > max_possible_exp
print(f"Experience exceeding (Age - 18) detected: {exceed_exp_mask.sum()}")
if exceed_exp_mask.sum() > 0:
    df.loc[exceed_exp_mask, 'Experience_Years'] = max_possible_exp[exceed_exp_mask]
    print("Capped excessive Experience to (Age - 18)")


Invalid Age records detected (<18 or >65): 4
Sample invalid age records:
    Employee_ID      Full_Name  Age
11       EMP012  Sneha Agarwal  -25
44       EMP045  Simran Saxena  137
189      EMP190    Manish Bose  -45
229      EMP230  Naveen Pandey  149
Replaced invalid Age records with median: 43

Invalid Salary records detected (<= 0): 3
Sample invalid salary records:
    Employee_ID     Full_Name   Salary
27       EMP028    Amit Joshi -43000.0
98       EMP099   Aarav Reddy      0.0
174      EMP175  Varun Bhatia -48000.0
Replaced non-positive Salary records with median: $94,000.00

Negative Experience records detected: 2
Replaced negative Experience records with 0
Experience exceeding (Age - 18) detected: 12
Capped excessive Experience to (Age - 18)


---
## N. Check the Dataset Again After Cleaning

We now verify the integrity of the cleaned DataFrame to guarantee that:
1. All missing values have been resolved.
2. All duplicate records have been eradicated.
3. Data types match their analytical specifications.
4. Numerical distributions are bounded within valid domain ranges.


In [18]:
print("=== CLEANED DATASET INFO ===")
df.info()


=== CLEANED DATASET INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Employee_ID       400 non-null    str           
 1   Full_Name         400 non-null    str           
 2   Department        400 non-null    str           
 3   Age               400 non-null    int64         
 4   Gender            400 non-null    str           
 5   City              400 non-null    str           
 6   Joining_Date      400 non-null    datetime64[us]
 7   Salary            400 non-null    float64       
 8   Experience_Years  400 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(2), str(5)
memory usage: 28.3 KB


In [19]:
print("=== CLEANED NUMERICAL SUMMARY ===")
df.describe()


=== CLEANED NUMERICAL SUMMARY ===


,Age,Joining_Date,Salary,Experience_Years
count,400.000000,400,400.000000,400.000000
mean,41.885000,2020-01-21 16:12:00,93585.000000,16.952500
min,22.000000,2015-01-03 00:00:00,35000.000000,0.000000
25%,32.000000,2017-09-12 06:00:00,69000.000000,7.000000
50%,43.000000,2020-03-07 12:00:00,94000.000000,18.000000
75%,50.000000,2022-06-29 06:00:00,121000.000000,26.000000
max,60.000000,2024-12-19 00:00:00,150000.000000,35.000000
std,10.882456,NaN,31570.707133,10.893894


In [20]:
# Final nulls and duplicates assertion check
print(f"Total Null Values Remaining: {df.isnull().sum().sum()}")
print(f"Total Duplicate Rows Remaining: {df.duplicated().sum()}")
assert df.isnull().sum().sum() == 0, "Warning: Null values still exist!"
assert df.duplicated().sum() == 0, "Warning: Duplicate rows still exist!"
print("Verification Passed: 0 nulls and 0 duplicates confirmed!")


Total Null Values Remaining: 0
Total Duplicate Rows Remaining: 0
Verification Passed: 0 nulls and 0 duplicates confirmed!


---
## O. Compare Dataset Before and After Cleaning

A rigorous audit comparing key metrics before and after the pipeline illustrates the transformation clearly.


In [21]:
comparison_table = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Duplicate Rows',
        'Total Missing Values',
        'Joining_Date Type',
        'Salary Type',
        'Age Min / Max',
        'Salary Min / Max',
        'Unique Departments',
        'Unique Cities',
        'Unique Genders'
    ],
    'Before Cleaning (Raw)': [
        len(df_raw),
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        df_raw.isnull().sum().sum(),
        str(df_raw['Joining_Date'].dtype),
        str(df_raw['Salary'].dtype),
        f"{df_raw['Age'].min():.0f} / {df_raw['Age'].max():.0f}",
        f"{df_raw['Salary'].dropna().min()} / {df_raw['Salary'].dropna().max()} (raw)",
        df_raw['Department'].nunique(dropna=True),
        df_raw['City'].nunique(dropna=True),
        df_raw['Gender'].nunique(dropna=True)
    ],
    'After Cleaning': [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df.isnull().sum().sum(),
        str(df['Joining_Date'].dtype),
        str(df['Salary'].dtype),
        f"{df['Age'].min()} / {df['Age'].max()}",
        f"${df['Salary'].min():,.0f} / ${df['Salary'].max():,.0f}",
        df['Department'].nunique(),
        df['City'].nunique(),
        df['Gender'].nunique()
    ]
})

print("=== BEFORE VS AFTER CLEANING COMPARISON ===")
comparison_table


=== BEFORE VS AFTER CLEANING COMPARISON ===


,Metric,Before Cleaning (Raw),After Cleaning
0,Total Rows,420,400
1,Total Columns,9,9
2,Duplicate Rows,20,0
3,Total Missing Values,133,0
4,Joining_Date Type,str,datetime64[us]
5,Salary Type,str,float64
6,Age Min / Max,-45 / 149,22 / 60
7,Salary Min / Max,"$0 / -$48,000 (raw)","$35,000 / $150,000"
8,Unique Departments,20,5
9,Unique Cities,24,6


---
## P. Save the Final Cleaned Dataset

We export the finalized, clean DataFrame to `data/cleaned_dataset.csv`.
`index=False` prevents Pandas from writing the default integer row index into the CSV file.


In [22]:
# Define output path
cleaned_data_path = '../data/cleaned_dataset.csv' if os.path.exists('../data') else 'data/cleaned_dataset.csv'

# Ensure directory exists and write CSV
os.makedirs(os.path.dirname(cleaned_data_path), exist_ok=True)
df.to_csv(cleaned_data_path, index=False)

print(f"Cleaned dataset successfully saved to: {cleaned_data_path}")
print(f"File size on disk: {os.path.getsize(cleaned_data_path):,} bytes")


Cleaned dataset successfully saved to: ../data/cleaned_dataset.csv


File size on disk: 28,201 bytes


---
## Data Cleaning Summary

Below is the consolidated final audit report of the entire data cleaning process.


In [23]:
print("=" * 65)
print("                    DATA CLEANING SUMMARY")
print("=" * 65)
print(f"  * Number of original rows:          {len(df_raw)}")
print(f"  * Number of duplicate rows removed: {df_raw.duplicated().sum()}")
print(f"  * Number of missing values handled: {df_raw.isnull().sum().sum()}")
print(f"  * Number of columns processed:      {df.shape[1]}")
print(f"  * Number of final rows:             {len(df)}")
print(f"  * Final dataset status:             CLEAN, VALIDATED & READY FOR ML")
print("=" * 65)


                    DATA CLEANING SUMMARY
  * Number of original rows:          420


  * Number of duplicate rows removed: 20
  * Number of missing values handled: 133
  * Number of columns processed:      9
  * Number of final rows:             400
  * Final dataset status:             CLEAN, VALIDATED & READY FOR ML
